[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/en_clase/S11_01_cnn_en_clase.ipynb)

# S11 · 01 — Redes convolucionales (CNN): cómo «ve» una red · versión en clase

⏱ 40 min en clase · Sprint 11 · UD 4.4

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos, y hoy nadie las usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Aquí lo verás con imágenes pequeñas que se ejecutan sin internet.

**Este cuaderno tiene 2 ejercicios** (1 a 2), los que se hacen en el aula. El pooling, la CNN completa en Keras frente a un MLP, mirar dentro de la red, el *data augmentation*, el mini-reto y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1 y 2 son los 2 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar los dígitos y verlos como matrices

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Qué forma tiene el conjunto y qué significa cada número?
2. ¿Qué valores puede tener un píxel y qué significan?
3. ¿Qué forma tendría un lote de fotos en color de 96×96?

In [ ]:
digitos = load_digits()
imagenes, etiquetas = digitos.images, digitos.target      # (1797, 8, 8) y (1797,)
print("Forma del conjunto:", imagenes.shape, "· valores de", imagenes.min(), "a", imagenes.max())
print("Un dígito (matriz 8×8):\n", imagenes[0].astype(int))

fig, ejes = plt.subplots(2, 8, figsize=(12, 3.4))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(imagenes[i], cmap="gray_r"); eje.set_title(f"es un {etiquetas[i]}", fontsize=9); eje.axis("off")
plt.suptitle("Dígitos 8×8 de scikit-learn (miniaturas de 64 píxeles)"); plt.tight_layout(); plt.show()

### ✏️ A3 · EJERCICIO 1 de 2 — Preparar los datos para una red

**Tu turno:** Prepara los datos para una red: en `X` guarda las imágenes **normalizadas entre 0 y 1** (divide entre 16) con forma `(n, 8, 8, 1)` (Keras espera un canal explícito). Después crea con `train_test_split` (`test_size=0.25`, `random_state=SEMILLA`, `stratify=etiquetas`) los conjuntos `X_ent, X_pru, y_ent, y_pru`.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Por qué se divide entre 16?
2. ¿Para qué sirve `np.newaxis` aquí?
3. ¿Cuántas imágenes quedan para entrenar y para la prueba?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: imagenes / 16.0 y añade el eje del canal con [..., np.newaxis]
X = ...
X_ent, X_pru, y_ent, y_pru = ...

assert X.shape == (1797, 8, 8, 1), "X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?"
assert X.max() <= 1.0 and X.min() >= 0.0, "Normaliza dividiendo entre 16 para que quede entre 0 y 1"
assert len(X_ent) + len(X_pru) == 1797 and len(X_pru) == 450, "Reparto 75 % / 25 %"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Programar la convolución a mano

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Qué tamaño tiene la salida de convolucionar una imagen de 14×14 con un filtro 3×3 sin relleno?
2. ¿Qué se hace en cada posición del filtro?
3. ¿Qué diferencia hay entre este filtro y los de una CNN?

In [ ]:
def convolucion2d(imagen, filtro):
    """Convolución 'válida' (sin relleno, paso 1) de una imagen 2D con un filtro 2D."""
    alto, ancho = imagen.shape
    k_alto, k_ancho = filtro.shape
    salida = np.zeros((alto - k_alto + 1, ancho - k_ancho + 1))
    for i in range(salida.shape[0]):
        for j in range(salida.shape[1]):
            zona = imagen[i:i + k_alto, j:j + k_ancho]
            salida[i, j] = np.sum(zona * filtro)
    return salida

### ✏️ A5 · EJERCICIO 2 de 2 — Calcular el tamaño de salida

**Tu turno:** Para una entrada de `n` píxeles, filtro de `k`, relleno (*padding*) `p` a cada lado y paso (*stride*) `s`, la salida mide `(n - k + 2p) / s + 1` (parte entera). Escribe `tam_salida(n, k, p=0, s=1)` que devuelva ese número entero. Sirve para entender por qué las imágenes "encogen" capa tras capa (y por qué Keras ofrece `padding="same"`).

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto da `tam_salida(8, 3)` y por qué?
2. ¿Qué tamaño tendría la salida de un filtro 5×5 sin relleno sobre una imagen de 32×32?
3. ¿Qué efecto tiene el relleno (`padding`)?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: aplica la fórmula con división entera (//)
def tam_salida(n, k, p=0, s=1):
    ...

assert tam_salida(8, 3) == 6, "8 píxeles con filtro 3 sin relleno dan 6"
assert tam_salida(8, 3, p=1) == 8, "Con relleno 1 el tamaño se conserva ('same')"
assert tam_salida(224, 7, p=3, s=2) == 112, "Caso típico de las primeras capas de ResNet: 224 → 112"
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Detectar bordes con un filtro (Sobel)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué bordes se iluminan con `filtro_vertical`?
2. ¿Cómo se obtiene el filtro horizontal a partir del vertical?
3. ¿Qué tamaño tiene la salida y por qué es menor que la imagen?

In [ ]:
# Imagen de prueba: rectángulo claro sobre fondo oscuro
prueba = np.zeros((14, 14)); prueba[3:11, 4:10] = 1.0
filtro_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])     # detecta bordes verticales (Sobel)
filtro_horizontal = filtro_vertical.T                                  # detecta bordes horizontales

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.6))
for eje, im, titulo in zip(ejes, [prueba, convolucion2d(prueba, filtro_vertical), convolucion2d(prueba, filtro_horizontal)],
                           ["Imagen original", "Bordes verticales", "Bordes horizontales"]):
    eje.imshow(im, cmap="gray"); eje.set_title(titulo); eje.axis("off")
plt.tight_layout(); plt.show()
print("Tamaño de la salida:", convolucion2d(prueba, filtro_vertical).shape, "(14 - 3 + 1 = 12)")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/2_Practica/para_casa/S11_01_cnn.ipynb)** trae el pooling, la CNN completa en Keras, los filtros y mapas de activación, el *data augmentation*, el mini-reto y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `matplotlib`, `tensorflow`, `keras` y, de `scikit-learn`, el conjunto de dígitos, la separación de datos y las métricas.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow.
    - `keras.utils.set_random_seed(SEMILLA)` fija las semillas para que los resultados se repitan.
- **Qué debe salir:**
    - `Listo ✔`. Importar TensorFlow tarda unos segundos.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'np' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 1. Una imagen es una matriz de números

#### A2 · Ejemplo — Cargar los dígitos y verlos como matrices

**Contexto TurisData.** El cliente tiene miles de fotos de alojamientos, zonas comunes y platos que hoy nadie usa. Antes de clasificarlas necesitas entender cómo una red **procesa una imagen**. Lo verás con imágenes pequeñas (dígitos 8×8 de `scikit-learn`) que se ejecutan sin internet.

**Idea.** Una imagen en escala de grises es una tabla de píxeles con valores (aquí de 0 = blanco, sin tinta, a 16 = negro, tinta máxima).
Una imagen en color tiene **3 canales** (rojo, verde, azul): una tabla por color. Su forma es
`(alto, ancho, canales)`. Un lote de imágenes añade una dimensión delante: `(n_imágenes, alto, ancho, canales)`.

- **Qué hace:**
    - `load_digits()` carga 1 797 imágenes de dígitos de 8×8 píxeles; `imagenes` son las matrices y `etiquetas` el dígito que representa cada una.
    - El primer `print` muestra la forma del conjunto y el rango de valores; el segundo, la matriz 8×8 de la primera imagen (`imagenes[0].astype(int)`).
    - El bucle dibuja con `imshow` las 16 primeras imágenes, cada una titulada con su etiqueta.
- **Qué debe salir:**
    - `Forma del conjunto: (1797, 8, 8) · valores de 0.0 a 16.0`, la matriz de la primera imagen (un 0, con valores de 0 a 15) y una rejilla de 16 dígitos en baja resolución.
- **Cómo leerlo:**
    - Para el ordenador, un dígito son **64 números** (de 0 = sin tinta a 16 = tinta máxima). Una foto en color tendría 3 canales: forma `(alto, ancho, 3)`.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué forma tiene el conjunto y qué significa cada número?* → `(1797, 8, 8)`: **1 797 imágenes** de **8 × 8** píxeles.
2. *¿Qué valores puede tener un píxel y qué significan?* → De **0** (sin tinta, blanco) a **16** (tinta máxima, negro).
3. *¿Qué forma tendría un lote de fotos en color de 96×96?* → `(n_imágenes, 96, 96, 3)`: se añade una dimensión delante (el lote) y los **3 canales** (rojo, verde, azul) detrás.

</details>

#### A3 · Ejercicio 1 — Preparar los datos para una red

- **Qué hace:**
    - `(imagenes / 16.0).astype("float32")[..., np.newaxis]` divide entre 16 (así los valores quedan entre 0 y 1) y añade un último eje de tamaño 1: el **canal** (forma `(1797, 8, 8, 1)`).
    - `train_test_split(X, etiquetas, test_size=0.25, random_state=SEMILLA, stratify=etiquetas)` separa un 25 % para la prueba (450 imágenes) y un 75 % para entrenar (1 347), con la misma proporción de cada dígito.
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `X debe tener forma (1797, 8, 8, 1): ¿has añadido el canal?` → falta `[..., np.newaxis]`.
    - `Normaliza dividiendo entre 16 ...` → los valores no están entre 0 y 1.
    - `Reparto 75 % / 25 %` → `test_size` no es 0.25 o no se han creado bien los cuatro conjuntos.
    - `NameError: name 'imagenes' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué se divide entre 16?* → Para **normalizar**: los píxeles pasan de 0–16 a **0–1**, una escala con la que las redes entrenan mejor.
2. *¿Para qué sirve `np.newaxis` aquí?* → Para añadir el **eje del canal** (`(8, 8)` → `(8, 8, 1)`): Keras espera una dimensión de canales, aunque sea una sola (escala de grises).
3. *¿Cuántas imágenes quedan para entrenar y para la prueba?* → **1 347** y **450**.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
X = (imagenes / 16.0).astype("float32")[..., np.newaxis]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, etiquetas, test_size=0.25, random_state=SEMILLA, stratify=etiquetas)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. La convolución: un filtro que se desliza

#### A4 · Ejemplo — Programar la convolución a mano

**Idea.** Un **filtro** (o *kernel*) es una mini-tabla, por ejemplo 3×3. Se coloca sobre una zona de la imagen,
se multiplica casilla a casilla, se suma y se obtiene **un** número. Después se desliza una posición y se repite.
El resultado es un **mapa de activación**: dice *dónde* aparece el patrón que busca el filtro (un borde vertical,
una esquina…). Una CNN **aprende sus propios filtros** durante el entrenamiento.

Vamos a hacerlo a mano con NumPy sobre una imagen sencilla: un rectángulo claro en fondo oscuro.

- **Qué hace:**
    - `convolucion2d(imagen, filtro)` calcula una convolución «válida» (sin relleno y con paso 1): crea una salida de tamaño `alto - k_alto + 1` por `ancho - k_ancho + 1`.
    - Dos bucles recorren cada posición; en cada una toma la zona de la imagen del tamaño del filtro, la multiplica casilla a casilla (`zona * filtro`) y suma todo (`np.sum`) para obtener **un** número.
- **Qué debe salir:**
    - No imprime nada: solo define la función.
- **Cómo leerlo:**
    - Es la operación que repite una capa convolucional miles de veces; la única diferencia es que la red **aprende** los números del filtro.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué tamaño tiene la salida de convolucionar una imagen de 14×14 con un filtro 3×3 sin relleno?* → **12×12** (14 − 3 + 1).
2. *¿Qué se hace en cada posición del filtro?* → Se **multiplica** la zona por el filtro casilla a casilla y se **suma**: sale un solo número.
3. *¿Qué diferencia hay entre este filtro y los de una CNN?* → Aquí los números del filtro los fijamos nosotros; en una CNN la red **los aprende** durante el entrenamiento.

</details>

#### A5 · Ejercicio 2 — Calcular el tamaño de salida

- **Qué hace:**
    - `tam_salida(n, k, p=0, s=1)` aplica la fórmula `(n - k + 2p) // s + 1`: `n` es el tamaño de la entrada, `k` el del filtro, `p` el relleno a cada lado y `s` el paso.
    - La comprobación prueba tres casos: 8 con filtro 3 → 6; con relleno 1 → 8 (se conserva, como `padding="same"`); y 224 con filtro 7, relleno 3 y paso 2 → 112 (las primeras capas de ResNet).
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Sin relleno, la imagen **encoge** en cada capa: por eso Keras ofrece `padding="same"`.
- **Si falla:**
    - `8 píxeles con filtro 3 sin relleno dan 6` → la fórmula está mal (falta el `+ 1` o el paréntesis).
    - `Con relleno 1 el tamaño se conserva` → falta `2 * p` (el relleno se pone a cada lado).
    - `Caso típico de las primeras capas de ResNet ...` → falta dividir entre `s` (usa `//`).
    - `NameError: name 'tam_salida' is not defined` → no has ejecutado la celda del ejercicio.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto da `tam_salida(8, 3)` y por qué?* → **6**: (8 − 3 + 0) // 1 + 1 = 6.
2. *¿Qué tamaño tendría la salida de un filtro 5×5 sin relleno sobre una imagen de 32×32?* → **28** (32 − 5 + 1).
3. *¿Qué efecto tiene el relleno (`padding`)?* → Con el relleno adecuado (p = 1 para filtro 3) la salida **conserva el tamaño** de la entrada (`padding="same"`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def tam_salida(n, k, p=0, s=1):
    return (n - k + 2 * p) // s + 1
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Detectar bordes con un filtro (Sobel)

- **Qué hace:**
    - `prueba` es una imagen 14×14 con un rectángulo claro (valor 1) sobre fondo oscuro (valor 0).
    - `filtro_vertical` es el filtro de Sobel, que detecta bordes verticales; `filtro_horizontal` es su traspuesta (`.T`) y detecta bordes horizontales.
    - Se dibujan tres imágenes: la original y su convolución con cada filtro (`convolucion2d`).
- **Qué debe salir:**
    - Tres imágenes: el rectángulo, los bordes verticales (se iluminan sus lados izquierdo y derecho) y los horizontales (arriba y abajo), y `Tamaño de la salida: (12, 12) (14 - 3 + 1 = 12)`.
- **Cómo leerlo:**
    - Cada filtro responde donde aparece su patrón: el vertical se activa en los lados izquierdo y derecho del rectángulo (con valores de signo contrario en cada lado) y no en la parte de arriba y abajo. El **mapa de activación** dice *dónde* está el patrón.
- **Si falla:**
    - `NameError: name 'convolucion2d' is not defined` → falta el ejemplo de la convolución a mano.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué bordes se iluminan con `filtro_vertical`?* → Los **laterales** (izquierdo y derecho) del rectángulo.
2. *¿Cómo se obtiene el filtro horizontal a partir del vertical?* → **Trasponiéndolo** (`filtro_vertical.T`).
3. *¿Qué tamaño tiene la salida y por qué es menor que la imagen?* → **12×12**: la imagen es de 14×14 y el filtro 3×3 sin relleno hace que la salida «encoja» 2 píxeles por lado.

</details>